# 02 — Feature Engineering & Cross-Asset Graph Signals

### Feature Architecture
We extract 80 dimensional features grouped into four distinct layers:
1. **Technical Indicators**: Moving Average Ratios (SMA5/SMA20), RSI (14), Bollinger Band Width, ATR (14).
2. **Institutional vs Retail Money Flow**: Net Haghighi/Hoghughi accumulation ratio, power of buyer/seller.
3. **Calendar & Seasonality**: Persian day of week (Saturday=0 to Wednesday=4), Jalali month, holiday proximity.
4. **Sector Graph Adjacency**: Correlation-weighted edge graphs connecting co-dependent industrial peers (e.g. Foolad to Fakhuz).


In [ ]:
import sqlite3
import pandas as pd
import numpy as np

DB_PATH = Path("../data/bourse.db")
conn = sqlite3.connect(DB_PATH)

df = pd.read_sql_query("""
    SELECT h.ins_code, h.gregorian_date, h.open, h.high, h.low, h.close, h.volume,
           c.vol_buy_retail, c.vol_sell_retail, c.vol_buy_institutional, c.vol_sell_institutional,
           c.power_retail, c.power_institutional
    FROM daily_history h
    LEFT JOIN client_type c ON h.ins_code = c.ins_code AND h.gregorian_date = c.gregorian_date
    WHERE h.ins_code = 1000 OR h.ins_code = (SELECT ins_code FROM instruments WHERE symbol='فولاد' LIMIT 1)
    ORDER BY h.gregorian_date ASC
    LIMIT 200
""", conn)
conn.close()

# Feature 1: Log returns
df['log_ret'] = np.log(df['close'] / df['close'].shift(1))

# Feature 2: Institutional Flow Ratio
df['net_ins_vol'] = df['vol_buy_institutional'].fillna(0) - df['vol_sell_institutional'].fillna(0)
df['total_vol'] = df['vol_buy_retail'].fillna(0) + df['vol_buy_institutional'].fillna(0)
df['ins_flow_ratio'] = df['net_ins_vol'] / df['total_vol'].replace(0, np.nan)

# Feature 3: Rolling True Range
df['high_low'] = df['high'] - df['low']
df['atr_14'] = df['high_low'].rolling(14).mean()

print(f"Engineered feature dataframe: {df.shape[0]} rows, {df.shape[1]} columns")
print(df[['gregorian_date', 'close', 'log_ret', 'ins_flow_ratio', 'atr_14']].tail(5))


### Feature Governance & Zero-Leakage Policy
All technical indicators, scalers, and graph adjacency laplacians are computed strictly using backward-looking windows ($t-k$ to $t$). No target information from $t+1$ is exposed to the feature matrix.
